# Support Vector Machine (SVM) Model Training

Train two SVM models for comparison:
- **BALANCED Model**: Trained on balanced data
- **RAW Model**: Trained on raw (unbalanced) data

Hyperparameters are tuned using GridSearchCV (10-fold Stratified CV, optimizing for ROC-AUC Macro).

**Note:** SVM requires feature scaling - StandardScaler is applied to all data.

## 1. Setup

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

# Define paths
BASE_PATH = Path('../../..').resolve()
DATA_PATH = BASE_PATH / 'data' / 'processed' / 'pickle'
MODELS_PATH = BASE_PATH / 'models' / 'svm'
RESULTS_PATH = BASE_PATH / 'data' / 'results' / 'svm'

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Paths configured")

✓ Paths configured


## 2. Load and Scale Data

In [2]:
# Load training data (BALANCED and RAW)
with open(DATA_PATH / 'X_train_balanced.pkl', 'rb') as f:
    X_train_balanced = pickle.load(f)
with open(DATA_PATH / 'y_train_balanced.pkl', 'rb') as f:
    y_train_balanced = pickle.load(f)

with open(DATA_PATH / 'X_train_raw.pkl', 'rb') as f:
    X_train_raw = pickle.load(f)
with open(DATA_PATH / 'y_train_raw.pkl', 'rb') as f:
    y_train_raw = pickle.load(f)


# Convert DataFrames to numpy arrays
if isinstance(X_train_balanced, pd.DataFrame):
    X_train_balanced = X_train_balanced.values
if isinstance(X_train_raw, pd.DataFrame):
    X_train_raw = X_train_raw.values

# Ensure targets are Series
if isinstance(y_train_balanced, pd.DataFrame):
    y_train_balanced = y_train_balanced.iloc[:, 0]
if isinstance(y_train_raw, pd.DataFrame):
    y_train_raw = y_train_raw.iloc[:, 0]

print(f"✓ Data loaded (before scaling)")
print(f"  BALANCED Training: {X_train_balanced.shape}")
print(f"  RAW Training: {X_train_raw.shape}")

# ============================================================================
# CRITICAL: Scale data (SVM requires feature scaling)
# ============================================================================
print(f"\n⚠️ Scaling features (CRITICAL for SVM)...")

scaler_balanced = StandardScaler()
X_train_balanced = scaler_balanced.fit_transform(X_train_balanced)

scaler_raw = StandardScaler()
X_train_raw = scaler_raw.fit_transform(X_train_raw)


print(f"✓ Data scaled successfully")
print(f"\nClass distribution:")
print(f"  BALANCED Train: {dict(pd.Series(y_train_balanced).value_counts().sort_index())}")
print(f"  RAW Train: {dict(pd.Series(y_train_raw).value_counts().sort_index())}")

✓ Data loaded (before scaling)
  BALANCED Training: (898, 651)
  RAW Training: (699, 651)

⚠️ Scaling features (CRITICAL for SVM)...
✓ Data scaled successfully

Class distribution:
  BALANCED Train: {0: np.int64(633), 1: np.int64(265)}
  RAW Train: {0: np.int64(633), 1: np.int64(66)}


## 3. Hyperparameter Tuning with GridSearchCV

In [3]:
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - GRID SEARCH")
print("="*70)
print("\nConfiguration:")
print("  • Metric: ROC-AUC Macro (10-fold Stratified CV)")
print("  • Cross-validation: StratifiedKFold (n_splits=10)")
print("  • Strategy: Tune C, gamma, kernel")
print("  • Kernel: RBF (non-linear, better for complex data)")

# Define CV strategy
cv_stratified = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define parameter grid for SVM
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1.0, 10.0, 100.0],           # Regularization strength
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],  # Kernel coefficient
    'degree': [2, 3],  # Para poly
}

def create_base_model():
    """Base SVM model with fixed parameters."""
    return SVC(
        kernel='rbf',              # Radial Basis Function (non-linear)
        class_weight='balanced',   # Handle class imbalance
        probability=True,         # Enable probability estimates (needed for ROC-AUC)
        random_state=42,
        verbose=0,
    )

def run_grid_search(X_train, y_train, model_name):
    """Execute GridSearchCV for BALANCED or RAW model."""
    print(f"\n{'-'*70}")
    print(f"Grid Search: {model_name} Model")
    print(f"{'-'*70}")
    
    grid_search = GridSearchCV(
        estimator=create_base_model(),
        param_grid=param_grid,
        cv=cv_stratified,
        scoring='recall_macro',
        n_jobs=-1,
        verbose=0,
    )
    
    print(f"Searching optimal hyperparameters...")
    print(f"(This may take a few minutes for SVM)")
    grid_search.fit(X_train, y_train)
    
    print(f"\n✓ Best hyperparameters ({model_name}):")
    for param, value in grid_search.best_params_.items():
        print(f"    {param}: {value}")
    
    print(f"\n✓ Best ROC-AUC (CV): {grid_search.best_score_:.4f}")
    
    # Save CV results
    cv_results = pd.DataFrame(grid_search.cv_results_)
    suffix = 'balanced' if 'BALANCED' in model_name else 'raw'
    results_path = RESULTS_PATH / f'gridsearch_results_{suffix}.csv'
    cv_results.to_csv(results_path, index=False)
    print(f"✓ Grid Search results saved")
    
    return grid_search.best_estimator_, grid_search

# Run grid search for both models
model_balanced, gs_balanced = run_grid_search(X_train_balanced, y_train_balanced, 'BALANCED')
model_raw, gs_raw = run_grid_search(X_train_raw, y_train_raw, 'RAW')

print(f"\n" + "="*70)
print(f"✓ Both SVM models trained successfully")
print(f"="*70)


HYPERPARAMETER TUNING - GRID SEARCH

Configuration:
  • Metric: ROC-AUC Macro (10-fold Stratified CV)
  • Cross-validation: StratifiedKFold (n_splits=10)
  • Strategy: Tune C, gamma, kernel
  • Kernel: RBF (non-linear, better for complex data)

----------------------------------------------------------------------
Grid Search: BALANCED Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
(This may take a few minutes for SVM)

✓ Best hyperparameters (BALANCED):
    C: 100.0
    degree: 3
    gamma: scale
    kernel: poly

✓ Best ROC-AUC (CV): 0.9847
✓ Grid Search results saved

----------------------------------------------------------------------
Grid Search: RAW Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
(This may take a few minutes for SVM)

✓ Best hyperparameters (RAW):
    C: 0.1
    degree: 2
    gamma: 0.001
    kernel: rbf

✓ Best ROC-AUC (CV): 0.6

## 4. Quick Training Validation

In [4]:
print("\n" + "="*70)
print("TRAINING SET PERFORMANCE")
print("="*70)

# BALANCED Model
y_train_pred_balanced = model_balanced.predict(X_train_balanced)
print(f"\nBALANCED Model:")
print(f"  Accuracy:  {accuracy_score(y_train_balanced, y_train_pred_balanced):.4f}")
print(f"  Precision: {precision_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  F1-Score:  {f1_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")

# RAW Model
y_train_pred_raw = model_raw.predict(X_train_raw)
print(f"\nRAW Model:")
print(f"  Accuracy:  {accuracy_score(y_train_raw, y_train_pred_raw):.4f}")
print(f"  Precision: {precision_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  F1-Score:  {f1_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")


TRAINING SET PERFORMANCE

BALANCED Model:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000

RAW Model:
  Accuracy:  0.6896
  Precision: 0.5840
  Recall:    0.7268
  F1-Score:  0.5593


## 5. Per-Class Metrics (Training Set)

In [5]:
print("\n" + "="*70)
print("PER-CLASS METRICS (TRAINING SET)")
print("="*70)

# BALANCED
prec_s, rec_s, f1_s, sup_s = precision_recall_fscore_support(y_train_balanced, y_train_pred_balanced, average=None)
print(f"\nBALANCED Model:")
df_s = pd.DataFrame({
    'Class': [0, 1],
    'Precision': prec_s,
    'Recall': rec_s,
    'F1-Score': f1_s,
    'Support': sup_s
})
print(df_s.to_string(index=False))

# RAW
prec_r, rec_r, f1_r, sup_r = precision_recall_fscore_support(y_train_raw, y_train_pred_raw, average=None)
print(f"\nRAW Model:")
df_r = pd.DataFrame({
    'Class': [0, 1],
    'Precision': prec_r,
    'Recall': rec_r,
    'F1-Score': f1_r,
    'Support': sup_r
})
print(df_r.to_string(index=False))


PER-CLASS METRICS (TRAINING SET)

BALANCED Model:
 Class  Precision  Recall  F1-Score  Support
     0        1.0     1.0       1.0      633
     1        1.0     1.0       1.0      265

RAW Model:
 Class  Precision   Recall  F1-Score  Support
     0   0.966368 0.680885  0.798888      633
     1   0.201581 0.772727  0.319749       66


## 6. Save Models and Scalers

In [6]:
# Save BALANCED model and scaler
model_path_balanced = MODELS_PATH / 'modelo_svm_balanced.pkl'
with open(model_path_balanced, 'wb') as f:
    pickle.dump(model_balanced, f)
print(f"✓ BALANCED model saved: {model_path_balanced}")

scaler_path_balanced = MODELS_PATH / 'scaler_svm_balanced.pkl'
with open(scaler_path_balanced, 'wb') as f:
    pickle.dump(scaler_balanced, f)
print(f"✓ BALANCED scaler saved: {scaler_path_balanced}")

# Save RAW model and scaler
model_path_raw = MODELS_PATH / 'modelo_svm_raw.pkl'
with open(model_path_raw, 'wb') as f:
    pickle.dump(model_raw, f)
print(f"✓ RAW model saved: {model_path_raw}")

scaler_path_raw = MODELS_PATH / 'scaler_svm_raw.pkl'
with open(scaler_path_raw, 'wb') as f:
    pickle.dump(scaler_raw, f)
print(f"✓ RAW scaler saved: {scaler_path_raw}")

print(f"\n✅ SVM models and scalers ready for evaluation")
print(f"\n⚠️ IMPORTANT: Use the saved scalers when making predictions on new data!")

✓ BALANCED model saved: /home/pablo/Desktop/Estressss/pdg/models/svm/modelo_svm_balanced.pkl
✓ BALANCED scaler saved: /home/pablo/Desktop/Estressss/pdg/models/svm/scaler_svm_balanced.pkl
✓ RAW model saved: /home/pablo/Desktop/Estressss/pdg/models/svm/modelo_svm_raw.pkl
✓ RAW scaler saved: /home/pablo/Desktop/Estressss/pdg/models/svm/scaler_svm_raw.pkl

✅ SVM models and scalers ready for evaluation

⚠️ IMPORTANT: Use the saved scalers when making predictions on new data!
